In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/moleculenet-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "partial-least-squares.py"

# 從 PCA 到 PCR 與 PLS

首次執行會下載實驗資料。可設定 DEEPCHEM_DATA_DIR 重用原始檔。要做練習，請在最後一格前新增以 `%%writefile -a "{script}"` 開頭的程式格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from sklearn.metrics import mean_absolute_error

SEED = 42
# ESOL: 1,128 筆水溶解度 log(mol/L)，DOI 10.1021/ci034243x。
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path.home() / ".cache" / "deepchem"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

def read_splits(loader):
    # 原始檔可共用，每次建立的特徵快取則彼此獨立。
    with tempfile.TemporaryDirectory() as scratch:
        tasks, splits, _ = loader(
            featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
            transformers=[], data_dir=str(cache), save_dir=scratch, reload=True)
        assert len(tasks) == 1 and all(np.all(s.w[:, 0] > 0) for s in splits)
        result = [(np.array([fingerprint(str(smi)) for smi in s.ids]),
                   s.y[:, 0].copy(), s.ids.copy()) for s in splits]
    print(tasks[0], "train / valid / test:", *(len(y) for _, y, _ in result))
    return result

from rdkit.Chem import rdFingerprintGenerator
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=512)

def fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(smiles)
    return generator.GetFingerprintAsNumPy(mol)


In [ ]:
%%writefile -a "{script}"
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

(xt, yt, _), (xv, yv, _), (xs, ys, _) = read_splits(dc.molnet.load_delaney)
# PCA：只用訓練指紋定出投影軸，再將新分子投影到同一組軸。
projection = make_pipeline(StandardScaler(), PCA(n_components=2, svd_solver="full"))
train_pc = projection.fit_transform(xt)
valid_pc = projection.transform(xv)
print("PC1 / PC2 explained variance (%):",
      np.round(100 * projection[-1].explained_variance_ratio_, 2))
print("First validation molecule, PC1 / PC2:", np.round(valid_pc[0], 3))

# PCR：沿用這兩欄座標，現在才加入訓練分子的實測溶解度。
regression = LinearRegression().fit(train_pc, yt)
print("Two-component PCR, first predicted log S:", round(regression.predict(valid_pc[:1])[0], 3))

counts = [1, 2, 3, 5, 10, 20]
fig, ax = plt.subplots(figsize=(6.5, 3.6), layout="constrained")
for name in ("PCR", "PLS"):
    models, scores = [], []
    for k in counts:
        if name == "PLS":
            model = PLSRegression(n_components=k, scale=True)
        else:
            model = make_pipeline(StandardScaler(), PCA(n_components=k, svd_solver="full"),
                                  LinearRegression())
        model.fit(xt, yt)
        scores.append(mean_absolute_error(yv, model.predict(xv).ravel()))
        models.append(model)
    print(name, "valid MAE", np.round(scores, 4))
    ax.plot(counts, scores, marker="o" if name == "PLS" else "s", label=name)
    if name == "PLS":
        selected = int(np.argmin(scores))
        pls = models[selected]
        print("selected PLS", counts[selected], "test MAE",
              round(mean_absolute_error(ys, pls.predict(xs).ravel()), 4))
ax.set(xticks=counts, xlabel="Number of components", ylabel="Validation MAE (log S)")
ax.legend()
fig.savefig(figure_dir / "09-09-partial-least-squares.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "09-09-partial-least-squares.png")))